# The HALD Knowledge Graph

HALD is a knowledge graph for human aging and longevity. It is described at this webpage on [Figshare](https://figshare.com/articles/dataset/HALD_a_human_aging_and_longevity_knowledge_graph_for_precision_gerontology_and_geroscience_analyses/22828196). This include versions in JSON and CSV. The CSV versions are structured for a particular package which we will not be using, so we will use the json packages which are more general.

First we'll create a data directory.

## Load the data in from the RDF created in the previous part

In [ ]:
from owlready2 import *
onto = get_ontology("./HALD.rdf").load()

## Exploring the nodes

* How many nodes are there?
* How many *types* of node are there (and what are they)?
* How many nodes of each type are there?

In [ ]:
# Nodes are the subjects and objects of triples, so you take both positions and combine them.

Nodes = list(default_world.sparql(         
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>
    SELECT DISTINCT ?node
    WHERE {
        ?node a ONTO:NamedIndividual .
    }
    """
)
)

print(f"There are {len(Nodes)} entities in the ontology")
for i in Nodes:
    print(i)

Alternatively we can do the counting in SPARQL if we don't want the nodes themselves

In [ ]:
# Nodes are the subjects and objects of triples, so you take both positions and combine them.

Nodes = list(default_world.sparql(         
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>

    SELECT (COUNT(DISTINCT ?node) AS ?numNodes)
    WHERE {
        ?node a ONTO:NamedIndividual .
    }
    """
)
)

print(f"There are {Nodes[0][0]} entities in the graph")


What about the *types* of node? We will need to specify that we also want the type of the object, and the the object is of type *Class*.

In [ ]:
# Nodes are the subjects and objects of triples, so you take both positions and combine them.

NodeTypes = list(default_world.sparql(         
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>

    SELECT DISTINCT ?type
    WHERE {
        ?node rdf:type ?type
        ?type a ONTO:Class
    }
    """
)
)

print(f"There are {len(NodeTypes)} types of node in the graph")
for i in NodeTypes:
    print(str(i[0]))

Let's now count how many of *each type* of node there are. To do this. we will need to`SELECT` the appropriate data, `GROUP` it `BY` type, and for convenience, sort (or `ORDER`) the final response `BY` the number of nodes.

In [ ]:
# Nodes are the subjects and objects of triples, so you take both positions and combine them.

NodeCounts = list(default_world.sparql(         
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>

    SELECT ?type (COUNT(DISTINCT ?node) AS ?count)
    WHERE {
        ?node rdf:type ?type
        ?type a ONTO:Class
    }
    GROUP BY ?type
    ORDER BY DESC(?count)
    """
)
)

print(f"There are {len(NodeCounts)} types of node in the graph")
for i in NodeCounts:
    print(i)

## Exploring the relations

Now let's do the same for the relations:

* How many relations (edges) are there?

To answer this, we can simply get the number of triples in the graph where the middle element is a relation.


In [ ]:
## What are the relations?
Relations = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>
    
    SELECT (COUNT(?relation) AS ?count)
    WHERE{  
       ?s ?relation ?o
       ?relation a ONTO:ObjectProperty
     }
     """))

print(f"There are {Relations[0][0]} relations in the ontology")


It's very easy to modify this to get the *types* of relation:

In [ ]:
## What are the relations?
RelationTypes = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>
    
    SELECT (COUNT(?relation) AS ?count)
    WHERE{  
       ?relation a ONTO:ObjectProperty
     }
     """))

print(f"There are {RelationTypes[0][0]} types of relation in the ontology")


This is quite typical of a knowledge graph that has been produced by large-scale text mining: there are a large number of relations because they have been inferred from natural language.

## Extracting the Ontology

Let us now extract the ontology. For this, we need to extract the triples, and then index them. This is much easier to do in python than in SPARQL.

In [ ]:
## What are the relations?
triples = list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.w3.org/2002/07/owl#>
    
    SELECT DISTINCT ?ts ?r ?to
    WHERE{  
       ?s ?r ?o
       ?r a ONTO:ObjectProperty
       ?s rdf:type ?ts
       ?o rdf:type ?to
       FILTER(?ts != owl:NamedIndividual)
       FILTER(?to != owl:NamedIndividual)
     }
     """))

print(f"There are {len(triples)} relations in the graph")

Let's look at a sampling of these to see what we are working with:

In [ ]:
for i in triples:
    print(i)

Build a table keyed by the start/end node types to keep track of how many type of relation there are between node types

In [ ]:
RelationTable = dict()
for t in triples:
    k = (t[0],t[2])
    if k not in RelationTable.keys():
        RelationTable[k] = 1
    else:
        RelationTable[k] += 1

print(RelationTable)

## Visualisation

Create and visualise the graph and its ontology. You will need to add the package `pyvis` for this. First, let's try out a small examples to see how it works.

In [ ]:
from pyvis.network import Network
net = Network()
net.add_node(0,title='a')
net.add_node(1,title='b')
net.add_node(2,title='c')
net.add_node(3,title='d')


net.add_edge(0,1,title='w')
net.add_edge(0,2,title='x')
net.add_edge(1,3,title='y')
net.add_edge(3,0,title='z')

net.toggle_physics(True)
net.repulsion()
net.show_buttons(filter_=['physics'])
net.save_graph("nx.html")

In [ ]:

    
net = Network()
for n in NodeTypes:
    net.add_node(str(n[0]))

for k in RelationTable.keys():
    net.add_edge(str(k[0]),str(k[1]),width=RelationTable[k]/100,title=RelationTable[k])

net.toggle_physics(True)
net.repulsion()
net.show_buttons(filter_=['physics'])
net.save_graph("HALD_Ontology.html")